# 00 · Dataset and integrity

SPEC 03, roadmap Stage 0 and 2.1. Builds the provisional file (`data/exploratory/provisional_train.parquet`) from the raw OpenF1 Parquets of the **train races only**, then checks its structure and integrity and recomputes the target of one race with independent plain code.

The file is rebuilt from scratch on every run of this notebook.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks" / "exploratory")]

from provisional import DATASET_PATH, build_train_dataset, train_races  # noqa: E402

from common.config import load_config  # noqa: E402
from ingestion.session import RAW_DIR  # noqa: E402

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
config = load_config()
print("seed", config.optuna.seed, "| test_fraction", config.split.test_fraction)

seed 42 | test_fraction 0.2


## 1. Build the file

It reads the train races only; the schedule (circuit of each race) comes from the cache.

In [2]:
DATASET_PATH.unlink(missing_ok=True)
df = build_train_dataset()
print(DATASET_PATH.relative_to(ROOT), DATASET_PATH.exists())
df.head(3)

data/exploratory/provisional_train.parquet True


,year,round,circuit,driver,team,race_pace_delta_pct,clean_laps,is_valid_row,quali_delta_pct,grid_position,reached_q3,fp_long_run_delta_pct,fp_tyre_deg,driver_pace_last3,team_pace_last3,teammate_quali_gap,team_pace_this_circuit_prev,circuit_type,track_temp,air_temp,rain_flag,sprint_pace_delta_pct,is_sprint_weekend,humidity,wind_speed
0,2023,1,Sakhir,VER,Red Bull Racing,-0.845271,52,True,0.000000,1,1,-1.237843,-0.029786,NaN,NaN,-0.153596,NaN,permanent,28.085263,23.88,0,NaN,0,21.652632,0.607368
1,2023,1,Sakhir,SAR,Williams,0.768059,51,True,2.167031,16,0,0.341474,0.313909,NaN,NaN,0.208832,NaN,permanent,28.085263,23.88,0,NaN,0,21.652632,0.607368
2,2023,1,Sakhir,NOR,McLaren,0.017271,47,True,1.864940,11,0,-0.188014,0.219427,NaN,NaN,-0.781750,NaN,permanent,28.085263,23.88,0,NaN,0,21.652632,0.607368


## 2. Train races and row counts

In [3]:
races = train_races()
held = sum(
    1
    for y in range(config.data.first_year, config.data.last_year + 1)
    for _ in (RAW_DIR / str(y)).glob("*/R/meta.json")
)
print(f"{held} races held, {len(races)} train races: {races[0]} to {races[-1]}")
per_year = df.groupby("year").agg(races=("round", "nunique"), rows=("driver", "size"), valid=("is_valid_row", "sum"))
per_year.loc["total"] = per_year.sum()
per_year

70 races held, 56 train races: (2023, 1) to (2025, 10)


,races,rows,valid
year,,,
2023,22,440,413
2024,24,478,451
2025,10,200,186
total,56,1118,1050


In [4]:
per_race = df.groupby(["year", "round"]).size()
print("drivers per race: min", per_race.min(), "max", per_race.max())
print(per_race.value_counts().sort_index().to_string())

drivers per race: min 19 max 20
19     2
20    54


## 3. Dtypes and memory

In [5]:
info = pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "nunique": df.nunique()})
print(f"{df.shape[0]} rows x {df.shape[1]} columns, {df.memory_usage(deep=True).sum() / 1e3:.0f} KB")
info

1118 rows x 25 columns, 432 KB


,dtype,non_null,nunique
year,int64,1118,3
round,int64,1118,24
circuit,object,1118,24
driver,object,1118,28
team,object,1118,10
race_pace_delta_pct,float64,1050,1047
clean_laps,int64,1118,71
is_valid_row,bool,1118,2
quali_delta_pct,float64,1108,1051
grid_position,Int64,1103,20


## 4. Duplicates

By record identity, `(year, round, driver)`, as the roadmap asks, and by whole row.

In [6]:
print("repeated (year, round, driver):", int(df.duplicated(["year", "round", "driver"]).sum()))
print("fully repeated rows:", int(df.duplicated().sum()))

repeated (year, round, driver): 0
fully repeated rows: 0


## 5. Constant and near-constant columns

Share of the most frequent value among the non-null ones; `rain_flag` and `reached_q3` are binary on purpose, so a high share there is a property of the world, not of the data.

In [7]:
top = pd.DataFrame(
    {
        c: {"unique": df[c].nunique(), "top_share": df[c].value_counts(normalize=True, dropna=True).iloc[0]}
        for c in df.columns
    }
).T
top.sort_values("top_share", ascending=False).head(8).round(3)

,unique,top_share
is_valid_row,2.0,0.939
rain_flag,2.0,0.768
is_sprint_weekend,2.0,0.750
circuit_type,3.0,0.625
reached_q3,2.0,0.511
year,3.0,0.428
team,10.0,0.100
clean_laps,71.0,0.067


## 6. Rows the valid-row rule drops

A row is valid with a qualifying time and at least 10 clean laps.

In [8]:
no_time = df["quali_delta_pct"].isna()
few_laps = df["clean_laps"] < 10
print("invalid rows:", int((~df["is_valid_row"]).sum()), "of", len(df))
print(pd.crosstab(no_time.rename("no quali time"), few_laps.rename("under 10 clean laps")))
print(
    "target empty exactly where the row is invalid:",
    bool((df["race_pace_delta_pct"].isna() == ~df["is_valid_row"]).all()),
)
print("valid rows with under 10 clean laps:", int((df["is_valid_row"] & few_laps).sum()))

invalid rows: 68 of 1118
under 10 clean laps  False  True 
no quali time                    
False                 1050     58
True                    10      0
target empty exactly where the row is invalid: True
valid rows with under 10 clean laps: 0


## 7. Hand check: 2024 round 8 (Monaco)

The target of every driver, recomputed with plain Python loops (no `provisional` function, no pandas arithmetic), straight from the rule in `DECISIONS.md`: lap above 1, not a pit-out lap, no pit record, a recorded time, and no overlap with a safety car, virtual safety car, red flag or track-wide yellow (opens at the flag or a "DEPLOYED" message, closes at the next track-scope CLEAR or GREEN). The field is the pooled clean laps of drivers with at least 10.

In [9]:
import statistics

race_dir = RAW_DIR / "2024" / "08" / "R"
laps = pd.read_parquet(race_dir / "laps.parquet").to_dict("records")
pitted = {(p["driver_number"], p["lap_number"]) for p in pd.read_parquet(race_dir / "pit.parquet").to_dict("records")}
messages = pd.read_parquet(race_dir / "race_control.parquet").sort_values("date").to_dict("records")

intervals, opened = [], None
for m in messages:
    on_track = m["scope"] == "Track"
    if opened is None:
        safety_car = m["category"] == "SafetyCar" and "DEPLOYED" in str(m["message"])
        if safety_car or (m["category"] == "Flag" and on_track and m["flag"] in ("RED", "YELLOW", "DOUBLE YELLOW")):
            opened = m["date"].timestamp()
    elif m["category"] == "Flag" and on_track and m["flag"] in ("CLEAR", "GREEN"):
        intervals.append((opened, m["date"].timestamp()))
        opened = None
if opened is not None:
    intervals.append((opened, float("inf")))
print("neutralised intervals (seconds apart):", [round(b - a) for a, b in intervals])

times = {}
for lap in laps:
    if lap["lap_number"] <= 1 or lap["is_pit_out_lap"] or (lap["driver_number"], lap["lap_number"]) in pitted:
        continue
    if pd.isna(lap["lap_duration"]) or pd.isna(lap["date_start"]):
        continue
    begin = lap["date_start"].timestamp()
    end = begin + lap["lap_duration"]
    if any(begin < b and end > a for a, b in intervals):
        continue
    times.setdefault(lap["driver_number"], []).append(lap["lap_duration"])

in_field = {d: t for d, t in times.items() if len(t) >= 10}
field_median = statistics.median([x for t in in_field.values() for x in t])
by_hand = {d: (statistics.median(t) / field_median - 1) * 100 for d, t in in_field.items()}
print(f"field median {field_median:.3f} s, {len(by_hand)} drivers with a target")

neutralised intervals (seconds apart): [1896]
field median 79.484 s, 16 drivers with a target


In [10]:
acronym = pd.read_parquet(RAW_DIR / "2024" / "08" / "Q" / "drivers.parquet").set_index("driver_number")["name_acronym"]
expected = {acronym[d]: v for d, v in by_hand.items()}
stored = df[(df["year"] == 2024) & (df["round"] == 8)].set_index("driver")["race_pace_delta_pct"]
check = pd.DataFrame({"dataset": stored, "by_hand": pd.Series(expected)})
check["diff"] = (check["dataset"] - check["by_hand"]).abs()
same_empty = bool((check["dataset"].isna() == check["by_hand"].isna()).all())
print("drivers:", len(check), "| empty targets match:", same_empty, "| max abs diff:", check["diff"].max())
assert same_empty and check["diff"].max() < 1e-9
check.round(6)

drivers: 20 | empty targets match: True | max abs diff: 0.0


,dataset,by_hand,diff
ALB,0.928489,0.928489,0.0
ALO,1.453123,1.453123,0.0
BOT,1.270067,1.270067,0.0
GAS,0.937296,0.937296,0.0
HAM,-1.225404,-1.225404,0.0
HUL,NaN,NaN,NaN
LEC,-1.237985,-1.237985,0.0
MAG,NaN,NaN,NaN
NOR,-1.364426,-1.364426,0.0
OCO,NaN,NaN,NaN


## Figures for the summary

In [11]:
print("No figure from this notebook goes into the summary: its content is tables (rows, integrity, hand check).")

No figure from this notebook goes into the summary: its content is tables (rows, integrity, hand check).


## Findings

**Nothing here changes a feature, a threshold or the preprocessing**, so this notebook adds no line to `DECISIONS.md`.

- 56 train races (2023 round 1 to 2025 round 10) and 1,118 rows: 440 in 2023, 478 in 2024 and 200 in 2025. Two races have 19 drivers, the rest 20.
- No repeated `(year, round, driver)` and no repeated row.
- No constant column. The most uniform ones are binary by nature (`rain_flag` 77% zeros, `is_sprint_weekend` 75% zeros), so none is a drop candidate.
- The valid-row rule drops 68 rows (6.1%): 58 have a qualifying time but fewer than 10 clean laps, and 10 have enough laps but no qualifying time. The target is empty exactly on those rows.
- The target of 2024 round 8 (Monaco) recomputed with plain code matches the dataset for all 20 drivers (maximum difference 0.0; 16 targets, 4 empty because those drivers crashed on lap 1).
- To check in the next notebooks: `quali_delta_pct` reaches 45.6% and `teammate_quali_gap` ±30–43% (outliers or errors, `02`); `grid_position` is empty in 15 rows of the file, 5 of them with a qualifying time and 2 among the training rows (`02`); the share of NaN in the practice, sprint and previous-circuit features (`02`).